# Longer curriculum training and repetition experiment
Generate a NEW source bundle with `python3 training/prepare_colab.py`, then upload it in setup. Select a GPU runtime. Existing checkpoints must be in the same Drive RUN_ROOT. The first repeated position is its second occurrence, including side to move.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io, os, subprocess, sys
uploaded = files.upload()
source = uploaded.get('colab-source.zip')
if source is None:
    raise ValueError('Upload colab-source.zip generated by training/prepare_colab.py')
root = Path('/content/Quoridoraner')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(source)) as archive:
    for entry in archive.infolist():
        target = (root / entry.filename).resolve()
        if not target.is_relative_to(root.resolve()):
            raise ValueError('Unsafe archive path')
    archive.extractall(root)
os.chdir(root)
# Keep Colab's CUDA-enabled PyTorch; do not install the Mac requirements pin.
import torch, numpy
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print('PyTorch', torch.__version__, 'CUDA', torch.version.cuda)
print(torch.cuda.get_device_properties(0))
subprocess.run([sys.executable, 'training/native/build_direct.py'], check=True)


## Persistent outputs
Mount Drive before training so checkpoints survive runtime deletion. Each run contains two models, AdamW states, replay, and search trees; larger runs can produce large checkpoints. Saves to Drive count toward runtime, so keep the same checkpoint interval across comparisons. Model benchmarks use local storage.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/quoridor-experiments')
RUN_ROOT.mkdir(parents=True, exist_ok=True)


## Find a useful model and batch size
Each case runs in a separate process. OOM cases are recorded and the sweep continues. Timings include input/output transfers for inference and replay sampling/transfers for updates. Memory is per model; the trainer keeps **two** models and optimizers resident, plus CPU replay/search trees. Leave substantial headroom. BF16 cases are skipped as unsupported on GPUs without BF16 support.


In [ ]:
subprocess.run([sys.executable, 'training/benchmark_cuda.py',
    '--out', str(RUN_ROOT / 'cuda-sweep.json'),
    '--architectures', 'small', 'residual64', 'residual128', 'residual256', 'residual512',
    '--batches', '32', '128', '512', '--precisions', 'fp32', 'bf16'], check=True)


In [ ]:
import json, pandas as pd
report = json.loads((RUN_ROOT / 'cuda-sweep.json').read_text())
print(report['hardware'])
pd.json_normalize(report['rows'])[['architecture', 'precision', 'batch', 'status',
    'parameters', 'inference.positions_per_second', 'update.positions_per_second',
    'update.peak_allocated_bytes']]


## Continue through the full curriculum and compare repetition cutoffs
Run 200 iterations for small/residual128/residual256. Existing threefold runs resume; separate repeat2 runs start fresh with the same settings and seed. Six experiments by default. Add seeds 2 and 3 for replication. All evaluation and hypothetical MCTS branches retain threefold repetition. Only actual training rollouts stop early, receiving zero value targets. Compare full-wall wins, cutoffs, game lengths and elapsed time. Early stopping can reinforce draws and requires this control experiment.


In [ ]:
architectures = ['small', 'residual128', 'residual256']
seeds = [1]  # expand to [1, 2, 3] after the pilot
iterations = 200
precision = 'bf16' if torch.cuda.is_bf16_supported() else 'fp32'
import signal, time
for limit in [3, 2]:
    for seed in seeds:
        for architecture in architectures:
            name = f'{architecture}-{precision}-seed{seed}' + ('' if limit == 3 else '-repeat2')
            directory = RUN_ROOT / name
            command = [sys.executable, '-u', 'training/train.py']
            if (directory / 'latest.pt').exists():
                command += ['--resume', str(directory), '--iterations', str(iterations)]
            else:
                settings = dict(out=directory, architecture=architecture, precision=precision,
                    training_repetition_limit=limit, iterations=iterations, seed=seed,
                    engine='direct', workers=min(4, os.cpu_count() or 1), games=128,
                    parallel_games=128, batch_size=256, updates=32, simulations=64,
                    curriculum='races', curriculum_interval=25, eval_games=20,
                    eval_every=10, heldout_games=40, checkpoint_seconds=120)
                for key, value in settings.items():
                    command += ['--' + key.replace('_', '-'), str(value)]
            command += ['--device', 'cuda', '--plain']
            print(' '.join(command), flush=True)
            last_display = 0
            with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                  text=True, bufsize=1) as process:
                try:
                    for line in process.stdout:
                        now = time.monotonic()
                        if now - last_display >= 2:
                            print(line, end='', flush=True)
                            last_display = now
                    if process.wait():
                        raise RuntimeError(f'Training failed: {directory}')
                except KeyboardInterrupt:
                    process.send_signal(signal.SIGINT)
                    for line in process.stdout:
                        print(line, end='', flush=True)
                    process.wait()
                    raise


## Review results
`summary.json` reports score, goal wins and artificial cutoffs separately; compare each arm against its matching baseline. `losses.json` reports fitting and `performance.json` reports throughput. Compare equal-work strength and elapsed time first. For equal-time comparisons, use separate fresh directories and the same external time budget per architecture; record actual games and updates because slower models complete less training. Early race curricula can yield high training scores without full-game strength.

For a frozen-target comparison independent of self-play data changes, upload a compatible `latest.pt` and use `training/benchmark_depth.py --phase frozen --source PATH --device cuda --architectures small residual64 residual128 --out PATH`. See `training/DEPTH_COMPARISON.md` for source/split requirements. That existing benchmark uses FP32. Keep precision identical within each comparison.

Ctrl-C/SIGINT in a terminal triggers a resumable save. A runtime being deleted may interrupt a save; Drive retains the last completed checkpoint. Colab's stop button is not a guarantee of graceful shutdown. Resume by rerunning the training cell after setup and Drive mounting.
